# Phase 10 — Production artifacts and final test

This notebook **does not evaluate the test set again**. The final test was opened exactly once by `src.package_production`, and the completed access record now blocks repeat scoring. Here we inspect the saved results, verify the bundle, and make one inference using a training-row verification sample.

## 1. Project setup

Run this notebook from the project root or from the `notebooks` folder.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PRODUCTION_SUMMARY_PATH, PRODUCTION_VERIFICATION_PATH
from src.package_production import verify_only
from src.predict import PriceRecommendationEngine

summary = json.loads(PRODUCTION_SUMMARY_PATH.read_text(encoding='utf-8'))
summary['phase']

## 2. Final validation-versus-test metrics

The candidate was frozen before these test metrics were calculated. Residual means `actual − predicted`, so a positive value means underprediction.

In [ ]:
validation = summary['validation_reference']
test = summary['final_test']['catboost_metrics']
pd.DataFrame([
    {'split': 'Validation', **validation},
    {'split': 'Final test', **test},
]).set_index('split')

## 3. Promotion decision and risk findings

Overall performance passes every rule fixed before the test. Segment warnings still matter: expensive, luxury, rare, and Low-confidence recommendations need stronger cautions in the app.

In [ ]:
display(pd.Series(summary['selection_decision']['checks'], name='passed'))
display(pd.json_normalize(summary['risk_findings'], sep='.').T.rename(columns={0: 'value'}))

## 4. Verify without touching the dataset

`verify_only()` checks every manifest hash, reloads the model, reproduces the stored sample prediction, and checks ZIP integrity. Its result explicitly records that zero test rows were accessed.

In [ ]:
verification_result = verify_only()
verification_result

## 5. Load the production engine and recommend one price

The stored verification sample is a training row, not a test row. The numeric recommendation is preserved separately from Indian-currency display text.

In [ ]:
verification = json.loads(PRODUCTION_VERIFICATION_PATH.read_text(encoding='utf-8'))
engine = PriceRecommendationEngine()
recommendation = engine.recommend(verification['sample'])
{
    'recommended_price': recommendation['recommended_price'],
    'recommended_price_display': recommendation['recommended_price_display'],
    'range': (recommendation['range_lower_display'], recommendation['range_upper_display']),
    'confidence': recommendation['confidence'],
    'confidence_reason': recommendation['confidence_reason'],
    'warnings': recommendation['warnings'],
}

## Phase 10 conclusion

Tuned CatBoost v1.0.0 is the final production model. The complete bundle contains the native model, exact feature order, category mappings, preprocessing medians, interval calibration, confidence profile, final metrics, metadata, model card, fresh-process verification, and a hash manifest. Phase 11 can now build the Streamlit interface around `PriceRecommendationEngine`.